# Example 0.13.1 / 1: Exercise 2-1: Dantzig's Diet Problem

Companion notebook for **0.13.1 Exercises on Building Models** in the AMPL lab Documentation, section "Exercise 2-1: Dantzig's Diet Problem". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 2-1: Dantzig's Diet Problem

Six foods have the calories, protein, calcium and vitamin A per pound shown below. Buy amounts that
meet at least the daily requirement of each nutrient at least cost. The data come from George
Dantzig's *Linear Programming and Extensions* (1963), with prices per pound from that time.

| | bread | meat | potatoes | cabbage | milk | gelatin | required |
|---|---|---|---|---|---|---|---|
| calories | 1254 | 1457 | 318 | 46 | 309 | 1725 | 3000 |
| protein (g) | 39 | 73 | 8 | 4 | 16 | 43 | 70 |
| calcium (mg) | 418 | 41 | 42 | 141 | 536 | 0 | 800 |
| vitamin A (I.U.) | 0 | 0 | 70 | 860 | 720 | 0 | 500 |
| cost per pound | 0.30 | 1.00 | 0.05 | 0.08 | 0.23 | 0.48 | |

The model is the diet model of The Diet Model (AMPL lab documentation, section 0.9.1) with only minimum requirements:
minimize $\sum_j c_j x_j$ subject to $\sum_j a_{ij} x_j \ge r_i$ for each nutrient $i$ and
$x_j \ge 0$. In the data and run fragments, `ampl` has already read the model.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NUTR;                         # nutrients
set FOOD;                         # foods
param cost {FOOD} > 0;            # dollars per pound
param amt {NUTR, FOOD} >= 0;      # nutrient units per pound of food
param required {NUTR} >= 0;       # daily requirement of each nutrient

var Buy {FOOD} >= 0;              # pounds of each food bought

minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];  # dollars per day

# each nutrient meets at least its requirement
subject to Need {i in NUTR}: sum {j in FOOD} amt[i,j] * Buy[j] >= required[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

foods = ["bread", "meat", "potatoes", "cabbage", "milk", "gelatin"]
amt = pd.DataFrame(
    [[1254, 1457, 318, 46, 309, 1725],
     [39, 73, 8, 4, 16, 43],
     [418, 41, 42, 141, 536, 0],
     [0, 0, 70, 860, 720, 0]],
    index=["calories", "protein", "calcium", "vitaminA"], columns=foods,
)
ampl.set["NUTR"] = list(amt.index)
ampl.set["FOOD"] = foods
ampl.param["cost"] = dict(zip(foods, [0.30, 1.00, 0.05, 0.08, 0.23, 0.48]))
ampl.param["amt"] = amt
ampl.param["required"] = {"calories": 3000, "protein": 70, "calcium": 800, "vitaminA": 500}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print({j: round(x, 4) for j, x in ampl.var["Buy"].to_dict().items() if x > 0})
print("cost per day:", round(ampl.obj["Total_Cost"].value(), 4))

HiGHS 1.14.0: optimal solution; objective 0.6196227853
2 simplex iterations
0 barrier iterations
solved
{'milk': 0.8154, 'potatoes': 8.6416}
cost per day: 0.6196


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set NUTR := calories protein calcium vitaminA;
set FOOD := bread meat potatoes cabbage milk gelatin;
param cost := bread 0.30  meat 1.00  potatoes 0.05  cabbage 0.08  milk 0.23  gelatin 0.48;
param amt:  bread  meat  potatoes  cabbage  milk  gelatin :=
calories     1254  1457       318       46   309     1725
protein        39    73         8        4    16       43
calcium       418    41        42      141   536        0
vitaminA        0     0        70      860   720        0;
param required := calories 3000  protein 70  calcium 800  vitaminA 500;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Buy, Total_Cost;

HiGHS 1.14.0: optimal solution; objective 0.6196227853
2 simplex iterations
0 barrier iterations
Buy [*] :=
   bread  0
 cabbage  0
 gelatin  0
    meat  0
    milk  0.815393
potatoes  8.64165
;

Total_Cost = 0.619623

